# Animated mode shapes — the spatial perturbation field

This notebook shows Nefes's **spatially-resolved mode shapes**: the continuous perturbation field
*inside* the ducts, animated over one oscillation cycle.

A `DUCT` is uniform and lossless, so its interior field is not approximated — it is the duct's own
phase relation (theory.md §12.3) evaluated at every interior station:

$$
f(s) = f_{\text{tail}}\,e^{-i\omega s/(\overline u+\overline c)},\quad
g(s) = g_{\text{head}}\,e^{-i\omega (L-s)/(\overline c-\overline u)},\quad
h(s) = h_{\text{tail}}\,e^{-i\omega s/\overline u}.
$$

The wave amplitudes $(f,g,h)$ at the duct faces already come out of every eigenmode and forced
solve, so the continuous shape is **exact post-processing** — no discretization, no extra solve.

**The API**

* `EigenmodeResult.animate_mode(i, variable="p")` — animate mode `i` (slider + play button).
* `EigenmodeResult.field_along_network(i, ...)` — the raw reconstructed field, one per root→leaf path.
* `PerturbationResponse.animate_field(freq, ...)` / `.field_along_network(freq, ...)` — same, forced.

**Reading the plot**

The x-axis is **developed length** (cumulative duct length from the inlet).
The animated line is the instantaneous physical perturbation $\Re\{\psi(x)\,e^{i\theta}\}$, framed by
its $\pm|\psi(x)|$ envelope.
Compact elements (area changes, the flame, terminals) are marked where the field may jump.
A branched network shows one trace per inlet→terminal path.

In [ ]:
import os, sys

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np

from nefes.shell import Network
from nefes.elements import catalog as cat
from nefes.thermo.configure import perfect_gas
from nefes.assembly.derive import ES_U, ES_C
from nefes.perturbation import PerturbationBC, eigenmodes, perturbation_response
from nefes.plotting import use_nefes_theme

# Plotly LaTeX rendering needs a MathJax shim in VSCode/Cursor notebooks.
from IPython.display import display, HTML
import plotly.offline as pyo
pyo.init_notebook_mode()
display(HTML(
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-AMS-MML_SVG"></script>'
))
_ = use_nefes_theme()

GAS = perfect_gas(287.0, 1.4)  # air


def mean_uc(sol, edge=0):
    """Mean axial velocity and sound speed on an edge of a converged solution."""
    t = sol.table()
    return float(t[ES_U, edge]), float(t[ES_C, edge])


def nearest_mode(res, f_target):
    """Index of the resolved mode closest to a target frequency [Hz]."""
    return int(np.argmin(np.abs(res.freqs - f_target)))

## 1. Organ pipe — a standing wave

A rigid–rigid (closed–closed) duct with no mean flow.
Its fundamental is the textbook $f_1 = \overline c/2L$, and the pressure mode shape is the standing
wave $p'(x)\propto\cos(\pi x/L)$ — antinodes at both rigid walls, a node at the centre.
Press **play** to watch it breathe.

In [ ]:
L = 0.5
net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=PerturbationBC.hard_wall()))
net.add(cat.duct(L))
net.add(cat.wall())
net.connect(0, 1, 0.05)
net.connect(1, 2, 0.05)
sol = net.solve()
assert sol.converged

u, c = mean_uc(sol)
res = eigenmodes(sol.problem, sol.x, (0.5 * c / (2 * L), 3.5 * c / (2 * L)))
print("modes [Hz]:", np.round(res.freqs, 1))

i1 = nearest_mode(res, c / (2 * L))
res.animate_mode(i1, variable="p", title="Organ-pipe fundamental: pressure standing wave").show()

The same mode in **velocity** is the complementary standing wave $u'(x)\propto\sin(\pi x/L)$:
nodes at the walls, an antinode at the centre — the rigid ends pin $u'=0$.

In [ ]:
res.animate_mode(i1, variable="u", title="Same mode in velocity: u' nodes at the walls").show()

## 2. Open–closed — a quarter-wave resonator

Replace the inlet wall with an ideal open end ($R=-1$).
Now the open inlet is a **pressure node** and the wall a pressure antinode:
$p'(x)\propto\sin(\pi x/2L)$, the quarter-wave mode at $f=\overline c/4L$.

In [ ]:
net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=PerturbationBC.open_end()))
net.add(cat.duct(L))
net.add(cat.wall())
net.connect(0, 1, 0.05)
net.connect(1, 2, 0.05)
sol = net.solve()
assert sol.converged

c = mean_uc(sol)[1]
res = eigenmodes(sol.problem, sol.x, (0.4 * c / (4 * L), 1.6 * c / (4 * L)))
iqw = nearest_mode(res, c / (4 * L))
print(f"quarter-wave mode: {res.freqs[iqw]:.1f} Hz  (c/4L = {c / (4 * L):.1f} Hz)")
res.animate_mode(iqw, variable="p", title="Open–closed quarter-wave: pressure node at the open end").show()

## 3. An expansion chamber — jumps at the area changes

A pipe → chamber → pipe network, quiescent and rigid at both ends.
Across each compact area change the **volume velocity** is continuous, so $u'$ **jumps** where the
area changes — the animation shows the velocity discontinuity at each junction, with the elements
marked.
(The pressure $p'$ stays continuous there; switch `variable="p"` to see it.)

In [ ]:
A1, A2 = 0.01, 0.05
Lp, Lc = 0.3, 0.4
net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=2.0)
net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=PerturbationBC.hard_wall()))
net.add(cat.duct(Lp))
net.add(cat.isentropic_area_change("expansion"))
net.add(cat.duct(Lc))
net.add(cat.isentropic_area_change("contraction"))
net.add(cat.duct(Lp))
net.add(cat.wall())
for a, b, ar in [(0, 1, A1), (1, 2, A1), (2, 3, A2), (3, 4, A2), (4, 5, A1), (5, 6, A1)]:
    net.connect(a, b, ar)
sol = net.solve()
assert sol.converged

res = eigenmodes(sol.problem, sol.x, (50.0, 700.0))
print("modes [Hz]:", np.round(res.freqs, 1))
imode = nearest_mode(res, 300.0)
res.animate_mode(imode, variable="u", title="Expansion chamber: u' jumps at the area changes").show()

## 4. Mean flow — a partly travelling wave

With subsonic mean flow and partially-reflecting ends the up- and down-running waves no longer have
equal amplitude, so the field is a **mixed standing/travelling** wave: the crest drifts downstream
rather than only oscillating in place.
The mode is also lightly damped, $\sigma<0$.

In [ ]:
net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
net.add(cat.total_pressure_inlet(130000.0, 300.0, perturbation_bc=PerturbationBC.reflection(0.6)))
net.add(cat.duct(L))
net.add(cat.pressure_outlet(101325.0, 300.0, perturbation_bc=PerturbationBC.reflection(0.3)))
net.connect(0, 1, 0.05)
net.connect(1, 2, 0.05)
sol = net.solve()
assert sol.converged
u, c = mean_uc(sol)
print(f"mean Mach M = {u / c:.3f}")
res = eigenmodes(sol.problem, sol.x, (0.4 * c / (2 * L), 1.6 * c / (2 * L)))
i1 = nearest_mode(res, c / (2 * L) * (1 - (u / c) ** 2))
print(f"mode: {res.freqs[i1]:.1f} Hz, growth {res.growth_rates[i1]:+.1f} 1/s")
res.animate_mode(i1, variable="p", title="Mean flow: a partly travelling pressure wave").show()

## 5. A forced field

The same reconstruction works for a **forced** response.
Drive the network with `perturbation_response`, then animate the spatial field at any swept
frequency with `animate_field` — here near the fundamental, driven from the inlet.

In [ ]:
freqs = np.linspace(0.4 * c / (2 * L), 1.6 * c / (2 * L), 31)
resp = perturbation_response(sol.problem, sol.x, freqs)
f_drive = res.freqs[i1]
resp.animate_field(f_drive, variable="p", title=f"Forced response near {f_drive:.0f} Hz").show()

## 6. Overlaying variables, bases, and modes

`animate_mode` (and `animate_field`) can put **several quantities on one figure**.
Pass a list of variables, a whole basis flavor, or a list of modes — each gets its own animated line and legend entry, normalized independently.

In [ ]:
# A fresh organ pipe (closed–closed, quiescent) with clean harmonics f_n = n·c/2L.
Lshow = 0.5
net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=PerturbationBC.hard_wall()))
net.add(cat.duct(Lshow))
net.add(cat.wall())
net.connect(0, 1, 0.05)
net.connect(1, 2, 0.05)
sol = net.solve()
assert sol.converged

c = mean_uc(sol)[1]
res = eigenmodes(sol.problem, sol.x, (0.5 * c / (2 * Lshow), 2.6 * c / (2 * Lshow)))
print("modes [Hz]:", np.round(res.freqs, 1))
i1 = nearest_mode(res, c / (2 * Lshow))  # fundamental
i2 = nearest_mode(res, c / Lshow)  # first overtone (2 f1)

# Pass a list of variables to overlay them — here p' against u' for the fundamental.
# Each is normalized independently (different units), so both read O(1) on one axis.
res.animate_mode(i1, variable=["p", "u"], title="Fundamental: p' and u' overlaid (quadrature)").show()

### Choosing a basis

Instead of naming variables one by one, pick a **basis** flavor and `animate_mode` overlays its three components at once.
The flavors mirror `plot_mode`: `"primitive"` $(p'/\overline\rho\,\overline c,\ u',\ \rho'\,\overline c/\overline\rho)$, `"char"` $(f,g,h)$, `"network"` $(\dot m', p', h_t')$, and more (see `BASIS_LABELS`).
For a quiescent standing wave the primitive triplet shows pressure and density in phase, with velocity in quadrature.

In [ ]:
res.animate_mode(i1, basis="primitive", title="Fundamental in the primitive basis").show()

### Overlaying modes — and a note on phase

Pass a **list of mode indices** to animate several at once.
Modes generally have different frequencies, so they share a common real-time axis: the **first** listed mode is the reference and completes exactly one cycle per loop, while each other mode advances at $f_k/f_{\text{ref}}$ and *beats* against it.
Here the first overtone runs at twice the fundamental's rate — watch their relative phase drift apart.
The `envelope=False` toggle drops the $\pm|\psi|$ background shading, which would otherwise overlap for several modes.

In [ ]:
res.animate_mode(
    [i1, i2],
    variable="p",
    envelope=False,
    title="Fundamental + first overtone: the overtone beats at 2×",
).show()

## Takeaways

* Mode shapes are reconstructed **analytically** inside each duct from the face wave-amplitudes —
  exact, and free of any discretization.
* `animate_mode` / `animate_field` give an interactive picture (slider + play) of the
  standing/travelling wave in any variable (`p`, `u`, `rho`, or a raw characteristic `f`/`g`/`h`).
* Overlay several quantities at once: a list of variables, a whole `basis` flavor, or a list of
  modes (animated on a shared real-time axis, so unequal-frequency modes beat); `envelope=False`
  drops the background shading.
* Compact elements appear as **jumps** with markers; a branched network is decomposed into
  inlet→terminal paths on a shared developed-length axis.
* `field_along_network` returns the raw per-path field for custom plotting or post-processing.